In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import sympy as sp
import proplot as pplt
from scipy.optimize import minimize
warnings.filterwarnings('ignore')
pplt.rc.update({'figure.dpi':100,'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
MODELSDIR  = CONFIGS['filepaths']['models']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELS     = CONFIGS['experiments']
NNCONFIG   = MODELS['nn']
SRCONFIG   = MODELS['sr']
TARGETVAR  = CONFIGS['variables']['target']
SPLIT      = 'valid'

statsfile = os.path.join(SPLITSDIR,'stats.json')
with open(statsfile,'r',encoding='utf-8') as f:
    STATS = json.load(f)
TP_MEAN = STATS[f'{TARGETVAR}_mean']
TP_STD  = STATS[f'{TARGETVAR}_std']
ZMIN    = (0.0 - TP_MEAN) / TP_STD
print(f'zmin = {ZMIN:.4f}')

In [ ]:
def load_kernel_weights(weightsfrom):
    wlist = []
    for seed in NNCONFIG['seeds']:
        wpath = os.path.join(WEIGHTSDIR,f'{weightsfrom}_{seed}_weights.nc')
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            wlist.append(wds['k'].load())
    return xr.concat(wlist,dim='seed').mean('seed')

def load_features(splitname,fieldvars,localvars,weightsfrom):
    filepath = os.path.join(SPLITSDIR,f'norm_{splitname}.h5')
    with xr.open_dataset(filepath,engine='h5netcdf') as ds:
        ntime = ds.sizes['time']
        nlat  = ds.sizes.get('lat',1)
        nlon  = ds.sizes.get('lon',1)
        y     = ds[TARGETVAR].transpose('time','lat','lon').values.ravel()
        cols  = {}
        if weightsfrom:
            nsig     = ds.sizes['sig']
            dsig     = ds['dsig'].values
            surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
            weights  = load_kernel_weights(weightsfrom)
            for var in fieldvars:
                da       = ds[var].transpose('time','lat','lon','sig')
                arr      = da.values.reshape(-1,nsig)
                w        = weights.sel(field=var).values
                weighted = arr * w[None,:] * dsig[None,:]
                if surfmask is not None:
                    weighted = weighted * surfmask
                cols[var] = weighted.sum(axis=1)
        else:
            for var in fieldvars:
                da        = ds[var]
                cols[var] = da.transpose('time','lat','lon').values.ravel()
        for var in localvars:
            da        = ds[var]
            cols[var] = da.transpose('time','lat','lon').values.ravel()
    X         = pd.DataFrame(cols)
    validmask = np.isfinite(X).all(axis=1).values & np.isfinite(y)
    return X[validmask].reset_index(drop=True), y[validmask]

def prettify_eq(expr_or_str, var_names=None):
    all_vars = var_names or ['rh','thetae','thetaestar','lf','shf','lhf']
    var_syms = {name: sp.Symbol(name) for name in all_vars}
    symbol_names = {
        var_syms.get('rh',sp.Symbol('rh')):               r'\widehat{RH}',
        var_syms.get('thetae',sp.Symbol('thetae')):       r'\widehat{\theta_e}',
        var_syms.get('thetaestar',sp.Symbol('thetaestar')): r'\widehat{\theta_e^*}',
        var_syms.get('lf',sp.Symbol('lf')):               r'\mathrm{LF}',
        var_syms.get('shf',sp.Symbol('shf')):             r'\mathrm{SHF}',
        var_syms.get('lhf',sp.Symbol('lhf')):             r'\mathrm{LHF}'}
    if isinstance(expr_or_str, str):
        ns = {**{name: sp.Symbol(name) for name in ['rh','thetae','thetaestar','lf','shf','lhf']},
              'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
              'sqrt':sp.sqrt,'exp':sp.exp,'log':sp.log,'abs':sp.Abs,'cos':sp.cos,'sin':sp.sin}
        try:
            expr = eval(expr_or_str,{'__builtins__':{}},ns)
        except Exception:
            return expr_or_str
    else:
        expr = expr_or_str
    try:
        for atom in list(expr.atoms(sp.Float)):
            expr = expr.subs(atom,sp.Float(f'{float(atom):.3g}'))
        return f'${sp.latex(expr,symbol_names=symbol_names)}$'
    except Exception:
        return str(expr_or_str)

def parse_and_build(eq_str, var_names):
    var_syms = {name: sp.Symbol(name) for name in var_names}
    ns = {**var_syms,
          'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
          'sqrt':sp.sqrt,'exp':sp.exp,'log':sp.log,'abs':sp.Abs,'cos':sp.cos,'sin':sp.sin}
    expr        = eval(eq_str,{'__builtins__':{}},ns)
    float_atoms = sorted(expr.atoms(sp.Float),key=float)
    if not float_atoms:
        return None
    param_syms  = [sp.Symbol(f'c{i}') for i in range(len(float_atoms))]
    expr_param  = expr
    for fa,ps in zip(float_atoms,param_syms):
        expr_param = expr_param.subs(fa,ps)
    all_syms    = [var_syms[n] for n in var_names] + param_syms
    f_eval      = sp.lambdify(all_syms,expr_param,modules='numpy')
    return dict(expr=expr,float_atoms=float_atoms,param_syms=param_syms,
                f=f_eval,var_syms=var_syms)

def optimize_constants(eq_str, var_names, X_train, y_train, X_valid, y_valid):
    parsed = parse_and_build(eq_str,var_names)
    if parsed is None:
        return None
    f,float_atoms,param_syms = parsed['f'],parsed['float_atoms'],parsed['param_syms']
    x0 = np.array([float(fa) for fa in float_atoms])

    def objective(params):
        args  = [X_train[n].values for n in var_names] + list(params)
        raw   = f(*args)
        pred  = np.maximum(np.asarray(raw,dtype=float),ZMIN)
        return float(np.nanmean((y_train - pred)**2))

    res = minimize(objective,x0,method='L-BFGS-B')

    def eval_mm(params,X):
        args  = [X[n].values for n in var_names] + list(params)
        raw   = np.asarray(f(*args),dtype=float)
        logz  = np.maximum(raw,ZMIN)
        return np.maximum(np.expm1(logz*TP_STD+TP_MEAN),0.0)

    ytrue_mm    = np.maximum(np.expm1(y_valid*TP_STD+TP_MEAN),0.0)
    ypred_orig  = eval_mm(x0,X_valid)
    ypred_optim = eval_mm(res.x,X_valid)

    def mse_mm(ypred):
        return float(np.nanmean((ytrue_mm-ypred)**2))
    def r2_mm(ypred):
        ssres = np.nansum((ytrue_mm-ypred)**2)
        sstot = np.nansum((ytrue_mm-np.nanmean(ytrue_mm))**2)
        return float(1-ssres/sstot)

    expr_optim = parsed['expr']
    for fa,c in zip(float_atoms,res.x):
        expr_optim = expr_optim.subs(fa,sp.Float(c))

    return dict(
        eq_str=eq_str, var_names=var_names,
        original_params=x0, optimized_params=res.x,
        mse_orig=mse_mm(ypred_orig), mse_optim=mse_mm(ypred_optim),
        r2_orig=r2_mm(ypred_orig),   r2_optim=r2_mm(ypred_optim),
        expr_orig=parsed['expr'], expr_optim=expr_optim,
        f=f, float_atoms=float_atoms,
        success=res.success)

In [ ]:
opt_results = {}
for runname,runconfig in SRCONFIG['runs'].items():
    csvpath = os.path.join(MODELSDIR,'sr',f'{runname}_equations.csv')
    if not os.path.exists(csvpath):
        print(f'No equations CSV for {runname!r}, skipping')
        continue
    fieldvars   = runconfig['fieldvars']
    localvars   = runconfig.get('localvars',[])
    weightsfrom = runconfig.get('weightsfrom')
    var_names   = fieldvars + localvars
    print(f'Loading features for {runname}...')
    X_train,y_train = load_features('train',fieldvars,localvars,weightsfrom)
    X_valid,y_valid = load_features(SPLIT, fieldvars,localvars,weightsfrom)
    eqdf  = pd.read_csv(csvpath)
    rows  = []
    for _,row in eqdf.iterrows():
        eq_str     = row['equation']
        complexity = int(row['complexity'])
        loss       = float(row['loss'])
        print(f'  complexity={complexity}: {eq_str}')
        opt = optimize_constants(eq_str,var_names,X_train,y_train,X_valid,y_valid)
        rows.append(dict(complexity=complexity,loss=loss,eq_str=eq_str,opt=opt))
    opt_results[runname] = rows
    print(f'  Done ({len(rows)} equations)')

In [ ]:
def plot_mse_r2_comparison():
    for runname,rows in opt_results.items():
        valid = [(r['complexity'],r['opt']) for r in rows if r['opt'] is not None]
        if not valid:
            print(f'No optimizable equations for {runname}')
            continue
        complexities  = [v[0] for v in valid]
        mse_orig      = [v[1]['mse_orig']  for v in valid]
        mse_optim     = [v[1]['mse_optim'] for v in valid]
        r2_orig       = [v[1]['r2_orig']   for v in valid]
        r2_optim      = [v[1]['r2_optim']  for v in valid]
        fig,axs = pplt.subplots(ncols=2,refwidth=3.5,refheight=2.5)
        axs[0].plot(complexities,mse_orig, color='blue6', marker='o',ms=5,lw=1.5,label='Original')
        axs[0].plot(complexities,mse_optim,color='green7',marker='o',ms=5,lw=1.5,label='Re-optimized')
        axs[0].format(title='MSE',xlabel='Complexity',ylabel='MSE (mm$^2$)',grid=True)
        axs[0].legend(loc='upper right',ncols=1)
        axs[1].plot(complexities,r2_orig, color='blue6', marker='o',ms=5,lw=1.5,label='Original')
        axs[1].plot(complexities,r2_optim,color='green7',marker='o',ms=5,lw=1.5,label='Re-optimized')
        axs[1].format(title='R$^2$',xlabel='Complexity',ylabel='R$^2$',grid=True)
        axs[1].legend(loc='lower right',ncols=1)
        fig.format(suptitle=f'{runname} — SciPy Constant Re-optimization (split={SPLIT})')
        pplt.show()

plot_mse_r2_comparison()

In [ ]:
def plot_rh_sensitivity():
    for runname,rows in opt_results.items():
        if 'rh' not in SRCONFIG['runs'][runname]['fieldvars']:
            continue
        valid = [r for r in rows if r['opt'] is not None and 'rh' in r['opt']['var_names']]
        if not valid:
            continue
        best     = min(valid,key=lambda r: r['loss'])
        opt      = best['opt']
        var_names = opt['var_names']
        fieldvars   = SRCONFIG['runs'][runname]['fieldvars']
        localvars   = SRCONFIG['runs'][runname].get('localvars',[])
        weightsfrom = SRCONFIG['runs'][runname].get('weightsfrom')
        X_valid,_ = load_features(SPLIT,fieldvars,localvars,weightsfrom)
        rh_vals   = np.linspace(float(X_valid['rh'].min()),float(X_valid['rh'].max()),300)
        X_sweep   = pd.DataFrame({v: np.full(300,float(X_valid[v].mean())) if v != 'rh' else rh_vals
                                   for v in var_names})
        def predict_mm(params):
            args  = [X_sweep[n].values for n in var_names] + list(params)
            raw   = np.asarray(opt['f'](*args),dtype=float)
            logz  = np.maximum(raw,ZMIN)
            return np.maximum(np.expm1(logz*TP_STD+TP_MEAN),0.0)
        ypred_orig  = predict_mm(opt['original_params'])
        ypred_optim = predict_mm(opt['optimized_params'])
        label_orig  = prettify_eq(opt['expr_orig'],  var_names)
        label_optim = prettify_eq(opt['expr_optim'], var_names)
        fig,ax = pplt.subplots(refwidth=4.5,refheight=2.5)
        ax.plot(rh_vals,ypred_orig, color='blue6', lw=2,label=f'Original: {label_orig}')
        ax.plot(rh_vals,ypred_optim,color='green7',lw=2,label=f'Re-optimized: {label_optim}')
        ax.format(title=f'{runname} — Best Equation RH Sensitivity (complexity={best["complexity"]})',
                  xlabel='Normalized RH (z-score)',ylabel='Precipitation (mm)',grid=True)
        ax.legend(loc='upper left',ncols=1)
        pplt.show()

plot_rh_sensitivity()